## Step 1: Setting up the environment

I am using google colab kernel for this notebook

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

# Confirm versions 
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# We will use CPU throughout — no GPU needed to understand the architecture
device = torch.device('cpu')
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


In [2]:
d_model     = 512
num_heads   = 8
d_ff        = 2048
num_layers  = 6
dropout     = 0.1
max_seq_len = 100
vocab_size  = 1000

## Step 2 : Input Embedding Class

**InputEmbedding: converts integer token IDs into 512-dimensional dense vectors and scales them by sqrt(d_model) so they are on the same magnitude as positional encodings.**

In [3]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)   # nn.Embedding is a lookup table 
        self.d_model = d_model             # The self. prefix means it belongs to this object and will be accessible in the forward method.

    def forward(self, x):   # Every nn.Module must have a forward method .This defines what happens to the data when it passes through this component. 
        # x shape coming in : (batch_size, seq_len)  — integer token IDs e.g (2,5) : 2 sentences, each with 5 tokens
        # x shape going out : (batch_size, seq_len, d_model) e.g (2,5,512) : each token is now represented by a 512-dimensional vector
        return self.embedding(x) * math.sqrt(self.d_model)  # the math.sqrt(d_model) scaling is multiplied to the embeddings to scale them up. This is followed from the original transformer paper.

In [4]:
# Test InputEmbedding
embed = InputEmbedding(vocab_size, d_model)

# Fake input: batch of 2 sentences, each 5 tokens long
# Each number is a token ID (integer between 0 and vocab_size-1)
dummy_tokens = torch.tensor([
    [4, 27, 103, 56, 8],   # sentence 1
    [9, 41, 7,  200, 3],   # sentence 2
])

print("Input shape  :", dummy_tokens.shape)   # (2, 5)

output = embed(dummy_tokens)
print("Output shape :", output.shape)          # (2, 5, 512)
print("Sample vector (first token, first sentence):", output[0][0][:5])

Input shape  : torch.Size([2, 5])
Output shape : torch.Size([2, 5, 512])
Sample vector (first token, first sentence): tensor([ 42.9439,  14.4107,  24.0643, -13.5933, -36.1080],
       grad_fn=<SliceBackward0>)


## Step 3 : Positional Encoding

**PositionalEncoding: adds fixed sine and cosine patterns to the embeddings so the model knows the position of each token in the sequence. Shape never changes — it only adds information to the values.**

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]       # x.size(1) returns the actual sequence length of the current input. In our test it is 5. So `self.pe[:, :5, :]` slices the first 5 rows from our 100-row PE matrix. Shape: `(1, 5, 512)`.

        return self.dropout(x)

In [6]:
## Positional Encoding test
pos_enc = PositionalEncoding(d_model, max_seq_len, dropout)

pe_output = pos_enc(output)

print("Input shape :", output.shape)
print("Output shape:", pe_output.shape)
print()
print("Before PE:", output[0][0][:5])
print("After PE :", pe_output[0][0][:5])

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])

Before PE: tensor([ 42.9439,  14.4107,  24.0643, -13.5933, -36.1080],
       grad_fn=<SliceBackward0>)
After PE : tensor([ 47.7154,  17.1230,   0.0000, -13.9925, -40.1200],
       grad_fn=<SliceBackward0>)


## Step 4 : Scaled Dot-Product Attention

In [7]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q shape: (batch_size, num_heads, seq_len, d_k)
    # K shape: (batch_size, num_heads, seq_len, d_k)
    # V shape: (batch_size, num_heads, seq_len, d_k)

    d_k = Q.size(-1) 

    # Step 1: compute raw attention scores
    # Q @ K.transpose(-2,-1) shape: (batch_size, num_heads, seq_len, seq_len)   ## K.transpose(-2, -1) swaps the last two dimensions of K. K shape was (2, 8, 5, 64). After transpose it becomes (2, 8, 64, 5)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2: apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # Step 3: softmax over last dimension
    attention_weights = F.softmax(scores, dim=-1)

    # Step 4: multiply by V
    # output shape: (batch_size, num_heads, seq_len, d_k)
    output = torch.matmul(attention_weights, V)

    return output, attention_weights

In [8]:
## Scaled Dot-Product Attention test
batch_size = 2
num_heads  = 8
seq_len    = 5
d_k        = d_model // num_heads   # 512 // 8 = 64

# Fake Q, K, V matrices
Q = torch.randn(batch_size, num_heads, seq_len, d_k)
K = torch.randn(batch_size, num_heads, seq_len, d_k)
V = torch.randn(batch_size, num_heads, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print("Q shape             :", Q.shape)
print("K shape             :", K.shape)
print("V shape             :", V.shape)
print("Output shape        :", output.shape)
print("Attention weights   :", weights.shape)
print()
print("Weights for first head, first sentence, first token:")
print(weights[0][0][0])
print()
print("Do weights sum to 1?", weights[0][0][0].sum().item())

Q shape             : torch.Size([2, 8, 5, 64])
K shape             : torch.Size([2, 8, 5, 64])
V shape             : torch.Size([2, 8, 5, 64])
Output shape        : torch.Size([2, 8, 5, 64])
Attention weights   : torch.Size([2, 8, 5, 5])

Weights for first head, first sentence, first token:
tensor([0.2631, 0.1189, 0.0149, 0.3818, 0.2212])

Do weights sum to 1? 0.9999998807907104


## Step 5 : Multi-Head Attention

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size = x.size(0)
        x = x.view(batch_size, -1, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))  # When you call self.W_q(Q), PyTorch does:
                                                # output = Q @ W_q.weight.T + W_q.bias
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)

        batch_size = attn_output.size(0)
        attn_output = attn_output.transpose(1, 2)
        attn_output = attn_output.contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(attn_output)

In [10]:
## Multi-Head Attention test
mha = MultiHeadAttention(d_model, num_heads)

x = torch.randn(2, 5, 512)

output = mha(x, x, x)
print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])


## Step 6 : Layer Normalisation

In [11]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta  = nn.Parameter(torch.zeros(d_model))
        self.eps   = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

In [12]:
# LayerNorm test
ln = LayerNorm(d_model)

x = torch.randn(2, 5, 512)
out = ln(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)
print("Input mean (first token) :", x[0][0].mean().item())
print("Output mean (first token):", out[0][0].mean().item())
print("Input std  (first token) :", x[0][0].std().item())
print("Output std  (first token):", out[0][0].std().item())

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])
Input mean (first token) : 0.037789348512887955
Output mean (first token): 7.450580596923828e-09
Input std  (first token) : 0.995513379573822
Output std  (first token): 1.0009773969650269


## Step 7 : Feed-Forward Network

In [13]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.linear1  = nn.Linear(d_model, d_ff)
        self.linear2  = nn.Linear(d_ff, d_model)
        self.dropout  = nn.Dropout(dropout)
        self.activation = nn.GELU()

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [14]:
# FeedForward test
ff = FeedForward(d_model, d_ff, dropout)

x = torch.randn(2, 5, 512)
out = ff(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])


## Step 8 : Encoder Layer

In [15]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn       = FeedForward(d_model, d_ff, dropout)
        self.norm1     = LayerNorm(d_model)
        self.norm2     = LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sublayer 1: self-attention + residual + norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        # Sublayer 2: FFN + residual + norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x

In [16]:
## EncoderLayer test
enc_layer = EncoderLayer(d_model, num_heads, d_ff, dropout)

x = torch.randn(2, 5, 512)
out = enc_layer(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])


## Step 9 : Encoder Stack

In [17]:
class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = LayerNorm(d_model)

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [18]:
# Encoder stack test
encoder = Encoder(d_model, num_heads, d_ff, dropout, num_layers)

x = torch.randn(2, 5, 512)
out = encoder(x)

print("Input shape :", x.shape)
print("Output shape:", out.shape)
print("Number of encoder layers:", len(encoder.layers))

Input shape : torch.Size([2, 5, 512])
Output shape: torch.Size([2, 5, 512])
Number of encoder layers: 6


## Step 10 : Decoder Layer

In [19]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.masked_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn  = MultiHeadAttention(d_model, num_heads)
        self.ffn         = FeedForward(d_model, d_ff, dropout)
        self.norm1       = LayerNorm(d_model)
        self.norm2       = LayerNorm(d_model)
        self.norm3       = LayerNorm(d_model)
        self.dropout     = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Sublayer 1: masked self-attention
        attn1 = self.masked_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn1))

        # Sublayer 2: cross-attention
        attn2 = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn2))

        # Sublayer 3: FFN
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))

        return x

In [20]:
# DecoderLayer test
dec_layer = DecoderLayer(d_model, num_heads, d_ff, dropout)

# Simulate encoder output and decoder input
enc_out = torch.randn(2, 5, 512)   # encoder output (source)
dec_in  = torch.randn(2, 4, 512)   # decoder input (target so far)

out = dec_layer(dec_in, enc_out)

print("Encoder output shape:", enc_out.shape)
print("Decoder input shape :", dec_in.shape)
print("Decoder output shape:", out.shape)

Encoder output shape: torch.Size([2, 5, 512])
Decoder input shape : torch.Size([2, 4, 512])
Decoder output shape: torch.Size([2, 4, 512])


## Step 11 : Decoder Stack

In [21]:
class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return self.norm(x)

In [22]:
# decoder stack test
decoder = Decoder(d_model, num_heads, d_ff, dropout, num_layers)

enc_out = torch.randn(2, 5, 512)
dec_in  = torch.randn(2, 4, 512)

out = decoder(dec_in, enc_out)

print("Output shape:", out.shape)
print("Number of decoder layers:", len(decoder.layers))

Output shape: torch.Size([2, 4, 512])
Number of decoder layers: 6


## Step 12 : Full Transformer

In [23]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads,
                 d_ff, dropout, num_layers, max_seq_len):
        super().__init__()

        # Input side
        self.src_embedding = InputEmbedding(src_vocab_size, d_model)
        self.src_pos_enc   = PositionalEncoding(d_model, max_seq_len, dropout)

        # Output side
        self.tgt_embedding = InputEmbedding(tgt_vocab_size, d_model)
        self.tgt_pos_enc   = PositionalEncoding(d_model, max_seq_len, dropout)

        # Encoder and Decoder
        self.encoder = Encoder(d_model, num_heads, d_ff, dropout, num_layers)
        self.decoder = Decoder(d_model, num_heads, d_ff, dropout, num_layers)

        # Final projection to vocabulary
        self.projection = nn.Linear(d_model, tgt_vocab_size)

        # Initialise weights
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        src = self.src_pos_enc(self.src_embedding(src))
        return self.encoder(src, src_mask)

    def decode(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        tgt = self.tgt_pos_enc(self.tgt_embedding(tgt))
        return self.decoder(tgt, enc_output, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        logits = self.projection(dec_output)
        return logits

## Step 13 : Test the complete transformer

In [24]:
# Build the complete model
model = Transformer(
    src_vocab_size = vocab_size,
    tgt_vocab_size = vocab_size,
    d_model        = d_model,
    num_heads      = num_heads,
    d_ff           = d_ff,
    dropout        = dropout,
    num_layers     = num_layers,
    max_seq_len    = max_seq_len
)

# Fake source and target token sequences
src = torch.randint(0, vocab_size, (2, 5))   # 2 sentences, 5 source tokens each
tgt = torch.randint(0, vocab_size, (2, 4))   # 2 sentences, 4 target tokens each

# Forward pass
logits = model(src, tgt)

print("Source shape     :", src.shape)
print("Target shape     :", tgt.shape)
print("Logits shape     :", logits.shape)
print()
print("Expected logits shape: (2, 4, 1000)")
print("  2    = batch size")
print("  4    = target sequence length")
print("  1000 = vocab size (score for each word)")
print()

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print()

# Verify the output makes sense — convert logits to predictions
predictions = logits.argmax(dim=-1)
print("Predicted token IDs:", predictions)
print("Predictions shape  :", predictions.shape)

Source shape     : torch.Size([2, 5])
Target shape     : torch.Size([2, 4])
Logits shape     : torch.Size([2, 4, 1000])

Expected logits shape: (2, 4, 1000)
  2    = batch size
  4    = target sequence length
  1000 = vocab size (score for each word)

Total parameters    : 45,677,544
Trainable parameters: 45,677,544

Predicted token IDs: tensor([[615, 850, 850, 903],
        [167, 850, 167, 878]])
Predictions shape  : torch.Size([2, 4])
